# Banco de Dados II - Análise Polymarket

## Análise de Eventos Prediction Markets usando BigQuery

Este notebook realiza uma análise exploratória de dados de eventos Polymarket usando Google BigQuery como Data Warehouse.

## Imports e Configuração

In [ ]:
# Imports principais
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Configurações de visualização
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Cliente BigQuery
client = bigquery.Client(project='even-continuity-441808-j0')
dataset_id = 'polymarket_events'
print('✅ BigQuery conectado com sucesso')

## 1. EXPLORAÇÃO DOS DADOS

### 1.1 Limpeza e Carregamento

In [ ]:
# Carregar dados de transações
query = """
SELECT 
  market_id,
  market_description,
  market_category,
  timestamp,
  price,
  volume,
  participant_id
FROM `even-continuity-441808-j0.polymarket_events.transactions`
WHERE DATE(timestamp) >= DATE_SUB(CURRENT_DATE(), INTERVAL 90 DAY)
LIMIT 100000
"""

df = client.query(query).to_dataframe()
print(f'Dados carregados: {len(df)} registros')
print(f'\nDimensões: {df.shape}')
print(f'\nTipos de dados:\n{df.dtypes}')

### 1.2 Análise de Nulos e Estatísticas

In [ ]:
# Verificar dados nulos
print('Valores Nulos por Coluna:')
print(df.isnull().sum())
print(f'\n% de dados completos: {(1 - df.isnull().sum().sum() / (len(df) * len(df.columns))) * 100:.2f}%')

# Primeiras linhas
print('\nPrimeiros 5 registros:')
df.head()

### 1.3 Estatísticas Descritivas

In [ ]:
print('ESTATÍSTICAS GERAIS')
print('='*60)
print(f'Período: {df["timestamp"].min()} até {df["timestamp"].max()}')
print(f'Total de Mercados: {df["market_id"].nunique()}')
print(f'Total de Categorias: {df["market_category"].nunique()}')
print(f'Total de Participantes: {df["participant_id"].nunique()}')
print(f'\nESTATÍSTICAS PREÇO (0-1 scale):')
print(df['price'].describe())
print(f'\nESTATÍSTICAS VOLUME:')
print(df['volume'].describe())

## 2. ANÁLISE EXPLORATÓRIA

### 2.1 Distribuição de Transações por Categoria

In [ ]:
# Volume por categoria
categoria_volume = df.groupby('market_category').agg({
    'volume': ['sum', 'count', 'mean'],
    'market_id': 'nunique'
}).round(2)

categoria_volume.columns = ['Volume Total', 'Transações', 'Volume Médio', 'Mercados']
categoria_volume = categoria_volume.sort_values('Volume Total', ascending=False)
print('Volume e Transações por Categoria:')
print(categoria_volume)

# Gráfico
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
categoria_volume['Volume Total'].plot(kind='bar', ax=axes[0], color='skyblue')
axes[0].set_title('Volume Total por Categoria', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Volume Total')
axes[0].tick_params(axis='x', rotation=45)

categoria_volume['Transações'].plot(kind='bar', ax=axes[1], color='lightcoral')
axes[1].set_title('Quantidade de Transações por Categoria', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Transações')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 2.2 Top 10 Mercados por Volume

In [ ]:
# Top mercados
top_mercados = df.groupby(['market_id', 'market_description']).agg({
    'volume': ['sum', 'count', 'mean', 'std'],
    'price': ['mean', 'min', 'max']
}).round(4)

top_mercados.columns = ['Volume Total', 'Transações', 'Volume Médio', 'Desvio Volume',
                         'Preço Médio', 'Preço Min', 'Preço Max']
top_mercados = top_mercados.sort_values('Volume Total', ascending=False).head(10)

print('\nTOP 10 MERCADOS POR VOLUME:')
print(top_mercados)

# Visualizar
plt.figure(figsize=(12, 6))
top_10_labels = [f"{desc[:30]}..." if len(desc) > 30 else desc 
                  for desc in top_mercados.index.get_level_values(1)]
plt.barh(range(len(top_mercados)), top_mercados['Volume Total'], color='steelblue')
plt.yticks(range(len(top_mercados)), top_10_labels)
plt.xlabel('Volume Total')
plt.title('Top 10 Mercados por Volume Total', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.3 Distribuição de Preços

In [ ]:
# Análise de preços
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histograma de preços
axes[0, 0].hist(df['price'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Distribuição de Preços', fontsize=11, fontweight='bold')
axes[0, 0].set_xlabel('Preço')
axes[0, 0].set_ylabel('Frequência')

# Box plot de preços
axes[0, 1].boxplot(df['price'])
axes[0, 1].set_title('Box Plot de Preços', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Preço')

# Distribuição de volume
axes[1, 0].hist(df['volume'], bins=50, color='lightcoral', edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Distribuição de Volume', fontsize=11, fontweight='bold')
axes[1, 0].set_xlabel('Volume')
axes[1, 0].set_ylabel('Frequência')

# Scatter: Preço vs Volume
axes[1, 1].scatter(df['price'], df['volume'], alpha=0.5, s=10, color='green')
axes[1, 1].set_title('Preço vs Volume', fontsize=11, fontweight='bold')
axes[1, 1].set_xlabel('Preço')
axes[1, 1].set_ylabel('Volume')

plt.tight_layout()
plt.show()

print(f'Correlação Preço-Volume: {df["price"].corr(df["volume"]):.4f}')

## 3. AGREGAÇÕES E ANÁLISES

### 3.1 Tendência Temporal de Volume

In [ ]:
# Volume por dia
df['data'] = pd.to_datetime(df['timestamp']).dt.date
volume_diario = df.groupby('data').agg({
    'volume': 'sum',
    'market_id': 'nunique',
    'participant_id': 'nunique'
}).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Volume por dia
axes[0].plot(volume_diario['data'], volume_diario['volume'], marker='o', linewidth=2, color='steelblue')
axes[0].fill_between(range(len(volume_diario)), volume_diario['volume'], alpha=0.3, color='steelblue')
axes[0].set_title('Volume Diário', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Data')
axes[0].set_ylabel('Volume')
axes[0].tick_params(axis='x', rotation=45)

# Mercados e participantes ativos
axes[1].plot(volume_diario['data'], volume_diario['market_id'], marker='s', label='Mercados', linewidth=2, color='darkgreen')
axes[1].plot(volume_diario['data'], volume_diario['participant_id'], marker='^', label='Participantes', linewidth=2, color='darkred')
axes[1].set_title('Mercados e Participantes Ativos por Dia', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Data')
axes[1].set_ylabel('Quantidade')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 3.2 Análise de Participantes

In [ ]:
# Distribuição de transações por participante
participante_stats = df.groupby('participant_id').agg({
    'volume': ['count', 'sum', 'mean'],
    'price': 'mean'
}).reset_index()

participante_stats.columns = ['participant_id', 'transacoes', 'volume_total', 'volume_medio', 'preco_medio']
participante_stats = participante_stats.sort_values('volume_total', ascending=False)

print(f'\nTOP 10 PARTICIPANTES:')
print(participante_stats.head(10))

# Gráfico de concentração
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top participantes
top_10_part = participante_stats.head(10)
axes[0].barh(range(len(top_10_part)), top_10_part['volume_total'], color='orange')
axes[0].set_yticks(range(len(top_10_part)))
axes[0].set_yticklabels(top_10_part['participant_id'])
axes[0].set_xlabel('Volume Total')
axes[0].set_title('Top 10 Participantes por Volume', fontsize=12, fontweight='bold')

# Distribuição de transações
axes[1].hist(participante_stats['transacoes'], bins=50, color='purple', edgecolor='black', alpha=0.7)
axes[1].set_title('Distribuição de Transações por Participante', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Número de Transações')
axes[1].set_ylabel('Frequência')

plt.tight_layout()
plt.show()

## 4. ALERTAS DE ANOMALIAS

### ALERTA 1: Movimentos de Preço Anômalos (>10% em 24h)

In [ ]:
# Executar query de alerta 1
query_alerta1 = """
WITH price_changes AS (
  SELECT 
    market_id,
    market_description,
    DATE(timestamp) as data,
    FIRST_VALUE(price) OVER (PARTITION BY market_id, DATE(timestamp) ORDER BY timestamp) as preco_inicio_dia,
    LAST_VALUE(price) OVER (PARTITION BY market_id, DATE(timestamp) ORDER BY timestamp ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) as preco_fim_dia
  FROM `even-continuity-441808-j0.polymarket_events.transactions`
  WHERE DATE(timestamp) >= DATE_SUB(CURRENT_DATE(), INTERVAL 7 DAY)
)
SELECT DISTINCT
  market_id,
  market_description,
  data,
  preco_inicio_dia,
  preco_fim_dia,
  ROUND(ABS(preco_fim_dia - preco_inicio_dia), 4) as variacao_absoluta,
  ROUND(100.0 * (preco_fim_dia - preco_inicio_dia) / NULLIF(preco_inicio_dia, 0), 2) as variacao_pct
FROM price_changes
WHERE ABS(100.0 * (preco_fim_dia - preco_inicio_dia) / NULLIF(preco_inicio_dia, 0)) > 10
ORDER BY data DESC, variacao_pct DESC
"""

alertas_preco = client.query(query_alerta1).to_dataframe()
print(f'🔴 ALERTAS DE PREÇO ENCONTRADOS: {len(alertas_preco)}')
if len(alertas_preco) > 0:
    print('\nTOP 10 MOVIMENTOS ANÔMALOS:')
    print(alertas_preco.head(10))
else:
    print('✅ Nenhum alerta de preço detectado nos últimos 7 dias')

### ALERTA 2: Volume Atípico (>2 desvios padrão)

In [ ]:
# Executar query de alerta 2
query_alerta2 = """
WITH volume_stats AS (
  SELECT 
    market_id,
    AVG(volume) as volume_medio,
    STDDEV(volume) as desvio_volume
  FROM `even-continutica-441808-j0.polymarket_events.transactions`
  WHERE DATE(timestamp) >= DATE_SUB(CURRENT_DATE(), INTERVAL 90 DAY)
  GROUP BY market_id
)
SELECT DISTINCT
  t.market_id,
  DATE(t.timestamp) as data,
  t.volume,
  ROUND(vs.volume_medio, 2) as volume_medio_historico,
  ROUND((t.volume - vs.volume_medio) / NULLIF(vs.desvio_volume, 0), 2) as z_score
FROM `even-continuity-441808-j0.polymarket_events.transactions` t
JOIN volume_stats vs ON t.market_id = vs.market_id
WHERE DATE(t.timestamp) >= DATE_SUB(CURRENT_DATE(), INTERVAL 7 DAY)
  AND ABS((t.volume - vs.volume_medio) / NULLIF(vs.desvio_volume, 0)) > 2
ORDER BY data DESC, z_score DESC
LIMIT 20
"""

alertas_volume = client.query(query_alerta2).to_dataframe()
print(f'🔴 ALERTAS DE VOLUME ENCONTRADOS: {len(alertas_volume)}')
if len(alertas_volume) > 0:
    print('\nTOP 10 PERÍODOS COM VOLUME ATÍPICO:')
    print(alertas_volume.head(10))
else:
    print('✅ Nenhum alerta de volume detectado nos últimos 7 dias')

## 5. RESUMO E CONCLUSÕES

In [ ]:
print('='*70)
print('RESUMO EXECUTIVO - ANÁLISE POLYMARKET')
print('='*70)
print(f'\nPeríodo Analisado: {df["timestamp"].min()} até {df["timestamp"].max()}')
print(f'\nMÉTRICAS GERAIS:')
print(f'  • Total de Transações: {len(df):,}')
print(f'  • Volume Total: {df["volume"].sum():,.2f}')
print(f'  • Valor Medio por Transação: ${df["volume"].mean():,.2f}')
print(f'  • Mercados Únicos: {df["market_id"].nunique()}')
print(f'  • Categorias: {df["market_category"].nunique()}')
print(f'  • Participantes: {df["participant_id"].nunique()}')
print(f'\nALERTAS DETECTADOS:')
print(f'  • Movimentos de Preço Anômalos (>10%): {len(alertas_preco)}')
print(f'  • Volume Atípico (>2σ): {len(alertas_volume)}')
print(f'\nCATEGORIAS MAIS ATIVAS:')
for idx, (cat, row) in enumerate(categoria_volume.head(3).iterrows(), 1):
    print(f'  {idx}. {cat}: {row["Volume Total"]:,.2f} volume, {int(row["Transações"])} transações')
print('\n' + '='*70)